# Lab 4: Cleaning II (Text, Dates, Encodings)

**DSA 405 · Week 4**

| | |
|---|---|
| **In class** | Friday, Sep 11 |
| **A4 due** | Thursday, Sep 17, 11:59 PM |
| **File** | `wolfpack_dining_raw.csv` |
| **Also this week** | **Bench Check 1** sign-up opens (slots run Weeks 5–7, in class) |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

Cleaning text is the slowest part of most wrangling work, and this week we take a look at
three common causes: strings that almost match, dates written in three different
formats, and text that was read with the wrong encoding somewhere and now shows `Ã©`
where an `é` should be. None of it is hard once you've seen the pattern, but each
one is slow the first time you encounter it, and that's normal.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

---
# Part 1: Explore (in class)

## Task 1.1: 27 strings, 6 categories

In Lab 3 we gave you a category canonicalizer ready-made (a canonicalizer is a
function that converts every variant spelling of a category to one standard name).
This week you build your own, and the first step is to read the raw values:

In [1]:
dining = load("wolfpack_dining_raw.csv", dtype=str)

for v in sorted(dining.category.dropna().unique()):
    print(repr(v))

NameError: name 'load' is not defined

Take your time with the `repr()` output. The quotes make leading and trailing
whitespace visible; `print` alone would not show it. As you can see, there are many issues we need to address: case (`COFFEE`/`coffee`), padding (`' Coffee'`, `'Coffee '`), punctuation
(`Fast-Casual`/`FastCasual`), synonyms (`C-Store`, `Coffee Shop`), and one Unicode
variant (`Café`).

Normalize first and map second. Mechanical normalization (strip, lowercase,
standardize punctuation and spacing) merges most of the variants on its own. The few
true synonyms left over need a hand-written map, and by that point the map is short
enough to check line by line:

In [ ]:
# Normalize and clean the 'category' column in the dining DataFrame
norm = (dining.category
        .str.strip()
        .str.lower()
        .str.replace("-", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True))

print("after normalizing :", norm.nunique(), "distinct")

# Define a dictionary of synonyms to standardize category names
SYNONYMS = {"c store": "convenience", "convenience store": "convenience",
            "coffee shop": "coffee", "café": "cafe",
            "fastcasual": "fast casual", "foodtruck": "food truck"}
dining["category_clean"] = norm.replace(SYNONYMS)

print("after mapping     :", dining.category_clean.nunique(), "distinct")
print(dining.category_clean.value_counts().to_dict())

unmapped = dining.category_clean.value_counts()
assert dining.category_clean.nunique() == 6, "still fragmented — check the map"
print("6 categories. The assert will catch it if a new variant ever sneaks in.")

## Task 1.2: Dates in three formats

Look at some values in `inspection_date`:

In [ ]:
print(dining.inspection_date.dropna().sample(8, random_state=405).tolist())

`03/20/2025`, `2024-07-18`, `January 4, 2025`, some with timestamps appended. Mixed
formats in one column usually mean the data was merged from two or more different
systems.

To **parse** a date means to read the text and convert it into a real date value.
`format="mixed"` lets each value be parsed by its own format. That is convenient, but
verify the result afterward, because the parser can read a value with the wrong format
and still finish without any error message:

In [ ]:
dates = pd.to_datetime(dining.inspection_date, format="mixed")

print("failed to parse:", dates.isna().sum())
print("range:", dates.min().date(), "to", dates.max().date())

assert dates.notna().all(), "some dates failed to parse"
assert dates.dt.year.isin([2024, 2025]).all(), "a year outside 2024-2025 appeared"
print("all dates parsed, all years in range")

Zero failures and a reasonable date range. Notice that we wrote asserts about the
years instead of only looking at them. (An **assert** is a line of code that states
something that must be true; Python stops with an error message if it is not.) A date
like `04/07/2025` can be read as April 7
or as July 4, and one of those readings is wrong with no error message to warn you. An
assertion about the outcome (the date range, the years, the pattern of weekdays) is
what catches a parse that used a different format than you intended.

## Task 1.3: Mojibake, the é that became Ã©

Print a few location names with `repr()`:

In [ ]:
damaged = dining[dining.location_name.str.contains("Ã", na=False)]
print(f"rows with Ã in location_name: {len(damaged)} of {len(dining)}")
print()
for v in damaged.location_name.head(4):
    print(repr(v))

62 rows. `CafÃ©` is `Café` that was written to the file as UTF-8 but read back as
Latin-1, so each 2-byte character split into two 1-byte characters. This pattern has a
name, **mojibake**: text that was read with the wrong encoding. Two facts about
mojibake are worth knowing. You can detect it, because the character `Ã` almost never
appears in real names, so seeing it is strong evidence of an encoding problem. And you
can repair it, because the original information is still in the file; it was only
decoded with the wrong encoding. You'll come back to these 62 rows in Task 2.4, where
you’ll be asked to think about the consequences of these mis-coded characters.

---
## Checkpoint: submit before leaving class

1. How many distinct category strings did mechanical normalization alone resolve, and how
   many needed the synonym map?
2. Which two assertions checked the parsed dates in Task 1.2, and what mistake would
   each one catch?
3. How many rows in `location_name` contain mojibake, and can the original text be
   recovered?

*Answers here.*

---
# Part 2: A4 (Text, Regex & Dates)

Four tasks. The first three are practice with today's tools. Task 2.4 needs the most
of your time, because it asks you to connect a technical choice to its consequence.
Explaining that connection (how an encoding decision changes what a report claims) is
the main skill this week teaches.

## Task 2.1: Location names, normalized

`location_name` has the same kinds of problems as `category`: capital and lowercase
letters mixed together, extra spaces at the start or the end of a name, and extra
spaces in the middle. Repair `location_name` with the same steps we used to repair
`category`. Normalize it, which here means strip the spaces off both ends, lowercase
the letters, and collapse repeated inner spaces into one. Leave the mojibake in
place; you will deal with it in Task 2.2.

Then report two things. First, how many distinct names there were before you
normalized and how many there are after. Second, find three cleaned names that came
from more than one raw spelling, and for each of the three, print the raw spellings
that became it. In this file every merge joins exactly two spellings, so you are
looking for three separate names, not three spellings of one name.

In [ ]:
# your normalization

## Task 2.2: Extraction with patterns

Three small jobs on this file using regular expressions. A **regular expression**
(regex) is a pattern that describes what a piece of text looks like, so that code can
find or extract every piece of text that matches it.

**a)** `avg_ticket` mixes `$24.87` with sentinels (`unknown`, `-999`, whitespace).
How many values in this column have a standard currency pattern? What
are the values that do not?

**b)** Extract the number from every currency value (use `.str.extract(r"([0-9.]+)")`,
then `to_numeric`) and state how many real prices we now have.

**c)** Count the rows whose `location_name` contains the mojibake character `Ã`.

In [ ]:
# your three patterns

## Task 2.3: Dates, with assumptions

Parse `inspection_date` like we did in class, then add at least three assertions: parse
completeness, year range, and one more (a good third: no date in the future). For each
assertion, add one clause naming the wrong parse it would catch (in other words, give an
example of a mistake that the assertion would catch).

In [ ]:
# your parse and assertions

## Task 2.4: The consequence of an encoding decision

A teammate proposes removing all rows that contain mojibake. Before agreeing or
objecting, measure what would change if those rows were removed.

1. Count rows per `campus_zone` before any drop. State which zones are smallest, and by
   how much.
2. Drop every row with mojibake in `location_name`, and count again.
3. Report the zone that changed most, and include specific numbers that support your statement (in other words, how much did the zone change when you dropped the rows).

Then write a paragraph about your findings. As you have seen, a choice about
text encoding has produced a false claim about where the university has dining
locations. Name the zone that changed the most (from part 3 above) and describe what the consequences were of dropping those rows. Finish
with what should happen to the mojibake rows instead of dropping them.

In [ ]:
# your before/after counts

*Paragraph here.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A4_[yourUnityID].ipynb`
4. Upload to the **A4** space on Moodle.

The **Checkpoint** section goes separately to **Week 4 In-Class Activity** before the
end of class on Friday. A4 is due **Thursday, Sep 17, 11:59 PM**.